# MVP04 — Camada Bronze

Materializa os 9 arquivos do volume `mvp.staging.kaggle_raw` como tabelas Delta
em `mvp.bronce`, **sem transformação de conteúdo**:
- Todas as colunas lidas como STRING (sem inferSchema);
- Nenhuma linha filtrada, deduplicada ou corrigida;
- Três colunas de controle adicionadas (prefixo `_`).

In [0]:
from pyspark.sql import functions as F

VOLUME = "/Volumes/mvp/staging/kaggle_raw"
DESTINO = "mvp.bronce"

ARQUIVOS = {
    "customers":            "olist_customers_dataset.csv",
    "geolocation":          "olist_geolocation_dataset.csv",
    "order_items":          "olist_order_items_dataset.csv",
    "order_payments":       "olist_order_payments_dataset.csv",
    "order_reviews":        "olist_order_reviews_dataset.csv",
    "orders":               "olist_orders_dataset.csv",
    "products":             "olist_products_dataset.csv",
    "sellers":              "olist_sellers_dataset.csv",
    "category_translation": "product_category_name_translation.csv",
}

In [0]:
def adding_bronze(tabela: str, arquivo: str) -> int:
    df = (spark.read.format("csv")
          .option("header", True)
          .option("inferSchema", False)   # tudo STRING
          .option("multiLine", True)      # comentários de review têm quebras de linha
          .option("escape", '"')          # aspas internas nos comentários
          .load(f"{VOLUME}/{arquivo}")
          .withColumn("_data_ingestao", F.current_timestamp())
          .withColumn("_arquivo_origem", F.col("_metadata.file_path"))
          .withColumn("_fonte", F.lit("Kaggle - Olist Brazilian E-Commerce")))

    (df.write.format("delta").mode("overwrite")
       .option("overwriteSchema", "true")
       .saveAsTable(f"{DESTINO}.{tabela}"))
    return spark.table(f"{DESTINO}.{tabela}").count()

resultados = {t: adding_bronze(t, a) for t, a in ARQUIVOS.items()}
for t, n in resultados.items():
    print(f"{t:22} {n:>10,} linhas")

In [0]:
import pandas as pd

ESPERADO = {
    "customers": 99441, "geolocation": 1000163, "order_items": 112650,
    "order_payments": 103886, "order_reviews": 99224, "orders": 99441,
    "products": 32951, "sellers": 3095, "category_translation": 71,
}

validacao = pd.DataFrame([
    {"tabela": t, "carregadas": resultados[t], "esperadas": ESPERADO[t],
     "status": "OK" if resultados[t] == ESPERADO[t] else "DIVERGENTE"}
    for t in ARQUIVOS
])
display(validacao)

In [0]:
%sql
COMMENT ON TABLE mvp.bronce.orders               IS 'Bronze. Cópia fiel de olist_orders_dataset.csv. Grão: um pedido.';
COMMENT ON TABLE mvp.bronce.order_items          IS 'Bronze. Cópia fiel de olist_order_items_dataset.csv. Grão: um item de pedido.';
COMMENT ON TABLE mvp.bronce.order_payments       IS 'Bronze. Cópia fiel de olist_order_payments_dataset.csv. Grão: uma transação de pagamento.';
COMMENT ON TABLE mvp.bronce.order_reviews        IS 'Bronze. Cópia fiel de olist_order_reviews_dataset.csv. Grão: uma avaliação.';
COMMENT ON TABLE mvp.bronce.customers            IS 'Bronze. Cópia fiel de olist_customers_dataset.csv. Grão: um customer_id.';
COMMENT ON TABLE mvp.bronce.products             IS 'Bronze. Cópia fiel de olist_products_dataset.csv. Grão: um produto.';
COMMENT ON TABLE mvp.bronce.sellers              IS 'Bronze. Cópia fiel de olist_sellers_dataset.csv. Grão: um vendedor.';
COMMENT ON TABLE mvp.bronce.geolocation          IS 'Bronze. Cópia fiel de olist_geolocation_dataset.csv. Grão: um ponto lat/lng.';
COMMENT ON TABLE mvp.bronce.category_translation IS 'Bronze. Cópia fiel de product_category_name_translation.csv. Referência PT → EN.';

In [0]:
spark.table("mvp.bronce.orders").printSchema()
display(spark.table("mvp.bronce.orders").limit(10))